# ENTREGA FINAL - VISUALIZACIÓN OBJETIVO SECUNDARIO OS03
## OS03 – prosperidad y productividad en Madrid. 
#### Caracterizar y explorar asociaciones entre exposición al ruido y prosperidad (nivel de ingresos).

### Se analizan asociaciones espaciales y temporales entre exposición al ruido e indicadores de renta/prosperidad.
Para ello se propone representaciones, análisis y correlaciones.

### DATOS UTILIZADOS:
* df_09* – INE, Atlas de Distribución de Renta de los Hogares (ADRH): cod_postal / seccion_censal / barrio, renta_media, renta_mediana, año
* Ruido:
* * df_02_cont_acustic → Ln, LAeq, por estación y año.
* * df_02_estac → información de las estaciones de medición
* * df_01_daily → agregados temporales
* Geografía: códigos postales / barrios usados en OS01 (df_actividad_cp) derivados de df_06_locales
* Normativa (contexto): df_03_noise_lim 

* PREPRARACIÓN DE ENTORNO

In [1]:
import altair as alt

# Desactivar vegafusion si quedó activado accidentalmente
alt.data_transformers.enable("default", max_rows=None)

# Renderer compatible con JupyterLab / Anaconda
alt.renderers.enable("mimetype")

RendererRegistry.enable('mimetype')

In [2]:
import pandas as pd
import altair as alt
import requests
import reverse_geocoder as rg
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.decomposition import PCA
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import unicodedata



pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

* CARGA DE DATOS PREPROCESADOS

In [3]:
# Carga los datos PREPROCESADOS disponibles en Kaggle. En la cuenta del alumno raquelahdo/ruido-datasets-procesados
# https://www.kaggle.com/datasets/raquelahdo/ruido-datasets-procesados

file_path_01_ruido_diario = "df_01_Ruido_diario_acumulado_processed.csv"
file_path_01_daily = "df_01_daily_processed.csv"
file_path_01_daily_plot = "df_01_daily_plot.csv"
file_path_02_cont_acustica = "df_02_contaminacion_acustica_processed.csv"
file_path_02_estac = "df_02_estacion_processed.csv"
file_path_03_noise_lim = "df_03_noise_limit_processed.csv"
file_path_05_est_ac_proc = "df_05_estaciones-acusticas_processed.csv"
# file_path_06_locales = "df_06_locales_terrazas_processed.csv"
# file_path_07_Mad_Q_vida = "df_07_Madrid-calidad-vida_processed.csv"
# file_path_07_Q_vida = "df_07_calidad-vida_processed.csv"
# file_path_07_evo_Q_vida = "df_07_evolucion-calidad-vida_processed.csv"
# file_path_07_evo_long_Q_vida = "df_07_evolucion-long-calidad-vida_processed.csv"
# file_path_07_fin_Q_vida_proc = "df_07_final-calidad-vida_processed.csv"
file_path_09_renta = "df_09_reduced_distribucion-renta_processed.csv"
file_path_os01_actividad_cp = "df_actividad_cp.csv"
file_path_os01_actividad_barrio = "df_actividad_barrio.csv"


df_01_ruido_diario = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_01_ruido_diario,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
df_01_daily = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_01_daily,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
df_01_daily_plot = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_01_daily_plot,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
df_02_cont_acustic = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_02_cont_acustica,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
df_02_estac = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_02_estac,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
df_03_noise_lim = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_03_noise_lim,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
df_05_est_ac = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_05_est_ac_proc,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
# df_06_locales = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_06_locales,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
# df_07_Mad_Q_vida = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_07_Mad_Q_vida,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
# df_07_Q_vida = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_07_Q_vida,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
# df_07_evo_Q_vida = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_07_evo_Q_vida,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
# df_07_evo_long_Q_vida = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_07_evo_long_Q_vida,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
# df_07_fin_Q_vida_proc = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_07_fin_Q_vida_proc,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
df_09_renta = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_09_renta,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
df_os01_actividad_cp = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_os01_actividad_cp,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )
df_os01_actividad_barrio = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "raquelahdo/ruido-datasets-procesados", file_path_os01_actividad_barrio,pandas_kwargs={"encoding": "utf-8-sig", "sep": ","} )


In [4]:
df_01_ruido_diario.head()

,NMT,year,month,day,periodo,laeq,l1,l10,l50,l90,l99,fecha,periodo_desc,mes_nombre,year_month,week,month_name
0,3,2014,1,1,D,57.4,66.6,61.1,54.3,49.1,45.0,2014-01-01,Diurno,Enero,2014-01,1,Enero
1,30,2014,1,1,D,55.6,64.7,59.2,51.9,45.9,43.5,2014-01-01,Diurno,Enero,2014-01,1,Enero
2,29,2014,1,1,T,57.2,67.4,62.1,49.0,36.6,29.8,2014-01-01,Total,Enero,2014-01,1,Enero
3,29,2014,1,1,N,55.0,66.7,57.4,41.5,30.9,29.2,2014-01-01,Nocturno,Enero,2014-01,1,Enero
4,29,2014,1,1,E,57.4,67.4,62.5,50.2,41.7,36.7,2014-01-01,Vespertino,Enero,2014-01,1,Enero


In [5]:
df_01_daily.head()

,fecha,periodo,laeq
0,2014-01-01,D,60.644
1,2014-01-01,E,61.176
2,2014-01-01,N,68.928
3,2014-01-01,T,66.156
4,2014-01-02,D,63.608


In [6]:
df_01_daily_plot.head()

,fecha,NMT,laeq
0,2014-01-01,3,61.050
1,2014-01-01,4,70.700
2,2014-01-01,5,64.475
3,2014-01-01,8,68.075
4,2014-01-01,10,64.450


In [7]:
df_02_cont_acustic.head()

,estacion,nombre,year,month,ld,le,ln,laeq,l1,l10,l50,l90,l99,fecha,mes_nombre,year_month,month_name
0,RF-19,Alto de Extremadura,1998,9,NaN,NaN,NaN,63.0,72.4,66.3,59.3,51.4,47.0,1998-09-01,Septiembre,1998-09,Septiembre
1,RF-16,Arturo Soria,1998,9,NaN,NaN,NaN,59.5,67.8,61.6,56.0,45.3,39.1,1998-09-01,Septiembre,1998-09,Septiembre
2,RF-14,Plaza Elíptica,1998,9,NaN,NaN,NaN,67.3,75.3,69.8,64.3,56.7,50.0,1998-09-01,Septiembre,1998-09,Septiembre
3,RF-11,Ramón y Cajal,1998,10,NaN,NaN,NaN,70.3,77.2,72.9,68.8,56.8,46.3,1998-10-01,Octubre,1998-10,Octubre
4,RF-14,Plaza Elíptica,1998,10,NaN,NaN,NaN,68.0,75.8,70.2,64.6,56.8,49.8,1998-10-01,Octubre,1998-10,Octubre


In [8]:
df_02_estac.head()

,estacion,laeq
0,RF-01,67.753311
1,RF-02,68.547328
2,RF-03,62.579321
3,RF-04,65.502980
4,RF-05,61.708411


In [9]:
df_03_noise_lim.head()

,noiselimitreportid_identifier,noisesource,limitvaluedefined,status,areatype,noiselevelindicator,limitvalue,explanation,limitvaluedefined_bool,noiselevel
0,LR_ES_00_01,allsources,yes,inforce,hospital,lday,60.0,This is not a limit value but rather an Acoust...,True,day
1,LR_ES_00_01,allsources,yes,inforce,hospital,levening,60.0,This is not a limit value but rather an Acoust...,True,evening
2,LR_ES_00_01,allsources,yes,inforce,hospital,lnight,50.0,This is not a limit value but rather an Acoust...,True,night
3,LR_ES_00_01,allsources,yes,inforce,school,lday,60.0,This is not a limit value but rather an Acoust...,True,day
4,LR_ES_00_01,allsources,yes,inforce,school,levening,60.0,This is not a limit value but rather an Acoust...,True,evening


In [10]:
df_05_est_ac.head()

,estacion,nombre,ubicacion,distrito,barrio,longitud,latitud,utm_x,utm_y,altitud,fecha_alta
0,RF-01,Paseo de Recoletos,Frente al n23 del Paseo de Recoletos,Centro,Justicia,-3.691877,40.422599,441307,4474893,648,2011-03-07
1,RF-02,Carlos V,"Plaza del Emperador Carlos V, junto al n1 del ...",Retiro,Jerónimos,-3.691509,40.409121,441327,4473397,629,1998-12-01
2,RF-03,Plaza del Carmen,Plaza del Carmen frente al n3 de la Calle de l...,Centro,Sol,-3.703175,40.419251,440346,4474529,657,1999-11-01
3,RF-04,Plaza de España,Frente al n18 de la Plaza de España,Moncloa - Aravaca,Argüelles,-3.712253,40.424005,439580,4475063,637,1998-12-01
4,RF-05,Barrio del Pilar,Parque de La Vaguada semiesquina de Avenida Mo...,Fuencarral - El Pardo,Pilar,-3.711543,40.478197,439688,4481078,673,1999-06-07


In [14]:
# df_06_locales.head()

In [ ]:
# df_07_Mad_Q_vida.head()

In [ ]:
# df_07_Q_vida.head()

In [ ]:
# df_07_evo_Q_vida.head()

In [ ]:
# df_07_evo_long_Q_vida.head()

In [ ]:
# df_07_fin_Q_vida_proc.head()

In [11]:
df_09_renta.head()

,secciones,periodo,municipio_codigo,municipio_nombre,renta_media
0,"2800401001 Álamo, El sección 01001",2018,28004,"Álamo, El",11.75
1,"2800401001 Álamo, El sección 01001",2019,28004,"Álamo, El",14.00
2,"2800401001 Álamo, El sección 01001",2020,28004,"Álamo, El",11.40
3,"2800401001 Álamo, El sección 01001",2021,28004,"Álamo, El",22.80
4,"2800401001 Álamo, El sección 01001",2022,28004,"Álamo, El",20.90


In [12]:
df_os01_actividad_cp.head()

,cod_postal_local,hora_cierre_media,num_locales,num_terrazas
0,28001.0,2.391304,5467,111
1,28002.0,1.340000,4873,159
2,28003.0,2.857143,4053,190
3,28004.0,1.512821,6274,151
4,28005.0,1.806122,5675,310


In [13]:
df_os01_actividad_barrio.head()

,barrio,hora_cierre_media,num_locales,num_terrazas
0,ABRANTES,15.870089,112,5
1,ACACIAS,16.429717,318,27
2,ADELFAS,18.316111,90,4
3,AEROPUERTO,9.560606,121,0
4,ALAMEDA DE OSUNA,15.987083,80,4


Los datos de renta proceden del Atlas de Distribución de Renta de los Hogares (INE) y están originalmente disponibles a nivel de sección censal. Para garantizar la coherencia espacial con los datos acústicos, la renta media se agregó a nivel municipal y anual, obteniendo una medida sintética comparable con los niveles de ruido nocturno.

In [24]:
#Creamos df_renta_municipio que es la renta media agregada a nivel MUNICIPAL y AÑO,
#lista para cruzarse con el ruido acústico.
#Se obtiene a partir de df_09_renta, con dos pasos:
# * Agregar por municipio y año
# * Eliminar la dimensión de sección censal, que no es compatible con ruido

# Se hace la media de las secciones censales del municipio
# Se obtiene una sola renta media por municipio y año

 #El ruido acústico está disponible a nivel municipal
# * Promediar secciones es práctica estándar en análisis socioeconómico
# * Es coherente con el enfoque descriptivo y no causal de OS03
# * Evita mezclar escalas incompatibles

df_renta_municipio = (
    df_09_renta.rename(columns={"periodo": "año"})
    .groupby( ["municipio_codigo", "municipio_nombre", "año"], as_index=False )
    .agg(renta_media=("renta_media", "mean"))
)

df_renta_municipio.head()

,municipio_codigo,municipio_nombre,año,renta_media
0,28004,"Álamo, El",2015,18.333333
1,28004,"Álamo, El",2016,18.333333
2,28004,"Álamo, El",2017,24.000000
3,28004,"Álamo, El",2018,15.000000
4,28004,"Álamo, El",2019,13.500000


In [16]:
df_os01_actividad_cp.head()

,cod_postal_local,hora_cierre_media,num_locales,num_terrazas
0,28001.0,2.391304,5467,111
1,28002.0,1.340000,4873,159
2,28003.0,2.857143,4053,190
3,28004.0,1.512821,6274,151
4,28005.0,1.806122,5675,310


In [17]:
df_os01_actividad_barrio.head()

,barrio,hora_cierre_media,num_locales,num_terrazas
0,ABRANTES,15.870089,112,5
1,ACACIAS,16.429717,318,27
2,ADELFAS,18.316111,90,4
3,AEROPUERTO,9.560606,121,0
4,ALAMEDA DE OSUNA,15.987083,80,4


Los datos acústicos disponibles no incluyen información territorial directa a nivel municipal, por lo que el análisis del OS03 se realiza mediante agregaciones anuales globales. Esta aproximación permite explorar asociaciones descriptivas entre la evolución del ruido nocturno medio y los indicadores de renta, manteniendo la coherencia metodológica sin introducir supuestos espaciales no soportados por los datos.

El análisis del OS03 a nivel de código postal se restringe al municipio de Madrid y se basa en la agregación de la renta procedente de secciones censales a códigos postales, con el objetivo de explorar asociaciones descriptivas entre prosperidad económica y exposición al ruido. Dada la naturaleza agregada de los datos y la ausencia de correspondencias directas, los resultados deben interpretarse como patrones socioambientales exploratorios, sin inferir relaciones causales.


In [ ]:
#Construimos df_os03_cp: el dataframe integrado final que contiene, por código postal y año en Madrid:
# cod_postal, año, renta_media (agregada a CP), ln_medio (ruido nocturno medio por CP)

# Renta por código postal en Madrid (df_09_renta → CP)








In [28]:
# Visualizaciones OS03 REPRESENTATIVAS (por código postal – Madrid
# OS03‑CP‑01 Scatter: Renta media vs ruido nocturno (Madrid, CP)
# Muestra: Asociación entre nivel de renta y exposición al ruido por código postal.
# Es la visualización clave de OS03 a escala urbana.

scatter_os03_cp01 = (
    alt.Chart(df_os03_cp)  # renta + ruido por CP
    .mark_circle(opacity=0.75)
    .encode(
        x=alt.X("renta_media:Q", title="Renta media (€)"),
        y=alt.Y("ln_medio:Q", title="Ruido nocturno medio (Ln) [dB]"),
        color=alt.Color(
            "ln_medio:Q",
            scale=alt.Scale(scheme="reds"),
            title="Ruido nocturno"
        ),
        tooltip=[
            alt.Tooltip("cod_postal:O", title="Código Postal"),
            alt.Tooltip("renta_media:Q", format=",.0f"),
            alt.Tooltip("ln_medio:Q", format=".1f")
        ]
    )
    .properties(
        width=650,
        height=420,
        title="OS03 · Renta y ruido nocturno por código postal (Madrid)"
    )
)

scatter_os03_cp01









NameError: name 'df_os03_cp' is not defined

✅ OS03‑CP‑02
Mapa: Desigualdad socioambiental en Madrid
🎯 Qué muestra
Distribución espacial conjunta de renta y ruido.
🧠 Justificación
La más intuitiva para “ver” desigualdad urbana.

In [ ]:
map_os03_cp02 = (
    alt.Chart(geojson_cp_madrid)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .encode(
        color=alt.Color(
            "renta_media:Q",
            scale=alt.Scale(scheme="blues"),
            title="Renta media (€)"
        ),
        tooltip=[
            alt.Tooltip("properties.cod_postal:O", title="CP"),
            alt.Tooltip("renta_media:Q", format=",.0f"),
            alt.Tooltip("ln_medio:Q", format=".1f")
        ]
    )
    .transform_lookup(
        lookup="properties.cod_postal",
        from_=alt.LookupData(
            df_os03_cp,
            key="cod_postal",
            fields=["renta_media", "ln_medio"]
        )
    )
    .project("mercator")
    .properties(
        width=700,
        height=650,
        title="OS03 · Renta y ruido por código postal en Madrid"
    )
)

map_os03_cp02

OS03‑CP‑03
Boxplot: renta según niveles de ruido
🎯 Qué muestra
Si los CP más ruidosos presentan peores niveles de renta.
🧠 Justificación
Resume desigualdad de forma muy clara.


In [ ]:
df_os03_cp["nivel_ruido"] = pd.qcut(
    df_os03_cp["ln_medio"],
    q=3,
    labels=["Bajo", "Medio", "Alto"]
)

box_os03_cp03 = (
    alt.Chart(df_os03_cp)
    .mark_boxplot()
    .encode(
        x=alt.X("nivel_ruido:N", title="Nivel de ruido nocturno"),
        y=alt.Y("renta_media:Q", title="Renta media (€)"),
        color=alt.Color("nivel_ruido:N", legend=None)
    )
    .properties(
        width=500,
        height=400,
        title="OS03 · Distribución de la renta según nivel de ruido (Madrid)"
    )
)

box_os03_cp03

OS03‑CP‑04
Evolución temporal: renta vs ruido (promedios CP)
🎯 Qué muestra
Tendencia temporal media dentro de Madrid.
🧠 Justificación
Añade dimensión temporal al OS03.

In [ ]:
df_cp_year = (
    df_os03_cp
    .groupby("año", as_index=False)
    .agg(
        renta_media=("renta_media", "mean"),
        ln_medio=("ln_medio", "mean")
    )
)

lines_os03_cp04 = alt.vconcat(
    alt.Chart(df_cp_year)
        .mark_line(point=True)
        .encode(x="año:O", y="ln_medio:Q")
        .properties(title="Ruido nocturno medio (Madrid)"),

    alt.Chart(df_cp_year)
        .mark_line(point=True)
        .encode(x="año:O", y="renta_media:Q")
        .properties(title="Renta media (Madrid)")
).resolve_scale(x="shared")

lines_os03_cp04

Preparación de la renta municipal (INE – df_09)
Hipótesis de los datos:
* Indicador: Renta media por unidad de consumo (€)
* Población: Total
* Nivel territorial: Municipio
* Variable temporal: periodo → año

In [25]:
# Preparación del ruido nocturno anual por municipio
# Métrica: Ln (ruido nocturno)
# Agregación: media anual
# Nivel territorial: municipio

# -------------------------------------------------
# RUIDO NOCTURNO ANUAL POR MUNICIPIO
# -------------------------------------------------

df_02_cont_acustic["fecha"] = pd.to_datetime(df_02_cont_acustic["fecha"])

df_ruido_ln = (
    df_02_cont_acustic
    .dropna(subset=["ln"])
    .assign(año=lambda d: d["fecha"].dt.year)
    [["estacion", "año", "ln"]]
)

# Transformación de estación a municipio
df_ruido_ln = df_ruido_ln.merge(
    df_02_estac[["estacion", "municipio_codigo"]],
    on="estacion",
    how="left"
)

df_ruido_ln = df_ruido_ln.dropna(subset=["municipio_codigo"])

# Agregación anual por municipio
df_ruido_municipio = (
    df_ruido_ln
    .groupby(["municipio_codigo", "año"], as_index=False)
    .agg(
        ln_medio=("ln", "mean")
    )
)

df_ruido_municipio.head()



KeyError: "['municipio_codigo'] not in index"

In [18]:
scatter_os03_01 = (
    alt.Chart(df_os03_merge)  # renta + ruido ya cruzados por territorio
    .mark_circle(opacity=0.7)
    .encode(
        x=alt.X("renta_media:Q", title="Renta media (€)"),
        y=alt.Y("ln_medio:Q", title="Ruido nocturno medio (Ln) [dB]"),
        color=alt.Color(
            "ln_medio:Q",
            scale=alt.Scale(scheme="reds"),
            title="Ruido nocturno"
        ),
        tooltip=[
            alt.Tooltip("territorio:N", title="Zona"),
            alt.Tooltip("renta_media:Q", title="Renta media", format=",.0f"),
            alt.Tooltip("ln_medio:Q", title="Ln [dB]", format=".1f")
        ]
    )
    .properties(
        width=650,
        height=420,
        title="OS03 · Renta media y exposición al ruido nocturno"
    )
)

scatter_os03_01

NameError: name 'df_os03_merge' is not defined

VISUALIZACIÓN OS03‑02
Mapa coroplético: renta vs ruido
🎯 Objetivo
Visualizar desigualdades territoriales de prosperidad y ruido.
📊 Tipo
Mapa coroplético + tooltip
🧠 Diseño

Color → renta media
Tooltip → renta + ruido nocturno
Unidad → código postal / barrio

💡 Justificación
Es la visualización más potente a nivel urbano para OS03.

In [ ]:
map_os03_02 = (
    alt.Chart(geojson_cp)   # geometría CP o barrios
    .mark_geoshape(stroke="white")
    .encode(
        color=alt.Color(
            "renta_media:Q",
            scale=alt.Scale(scheme="blues"),
            title="Renta media (€)"
        ),
        tooltip=[
            alt.Tooltip("cod_postal:O", title="Código Postal"),
            alt.Tooltip("renta_media:Q", format=",.0f"),
            alt.Tooltip("ln_medio:Q", format=".1f")
        ]
    )
    .transform_lookup(
        lookup="properties.cod_postal",
        from_=alt.LookupData(
            df_os03_merge,
            key="cod_postal",
            fields=["renta_media", "ln_medio"]
        )
    )
    .project("mercator")
    .properties(
        width=700,
        height=650,
        title="OS03 · Renta media y ruido nocturno por territorio"
    )
)

map_os03_02

VISUALIZACIÓN OS03‑03
Distribución de renta por niveles de ruido
🎯 Objetivo
Analizar si zonas más ruidosas presentan peores distribuciones de renta.
📊 Tipo
Boxplot o Violin plot
🧠 Diseño

X → nivel de ruido (bajo / medio / alto)
Y → renta media
Agrupación por percentiles de Ln

💡 Justificación
Resume muy bien desigualdad socio‑ambiental.

In [ ]:
box_os03_03 = (
    alt.Chart(df_os03_merge)
    .mark_boxplot()
    .encode(
        x=alt.X("nivel_ruido:N", title="Nivel de ruido nocturno"),
        y=alt.Y("renta_media:Q", title="Renta media (€)"),
        color=alt.Color("nivel_ruido:N", legend=None)
    )
    .properties(
        width=600,
        height=400,
        title="OS03 · Distribución de la renta según el nivel de ruido"
    )
)

box_os03_03

In [ ]:
VISUALIZACIÓN OS03‑04
Evolución temporal: renta vs ruido (Madrid)
🎯 Objetivo
Explorar si en el tiempo la evolución de la renta y del ruido siguen patrones comunes.
📊 Tipo
Small multiples / líneas sincronizadas
🧠 Diseño

Gráfico superior → ruido medio anual
Gráfico inferior → renta media
Eje tiempo compartido

💡 Justificación
Introduce dinámica temporal en OS03.

In [ ]:
lines_os03_04 = alt.vconcat(
    alt.Chart(df_ruido_anual)
        .mark_line(point=True)
        .encode(
            x="año:O",
            y="ln_medio:Q"
        )
        .properties(title="Ruido nocturno medio anual"),

    alt.Chart(df_renta_anual)
        .mark_line(point=True)
        .encode(
            x="año:O",
            y="renta_media:Q"
        )
        .properties(title="Renta media anual")
).resolve_scale(x="shared")

lines_os03_04

El análisis OS03 explora la relación entre la exposición al ruido ambiental y la prosperidad económica, medida a través de indicadores de renta. Las visualizaciones muestran asociaciones espaciales y temporales entre mayores niveles de ruido y menores niveles de renta en determinados territorios, evidenciando patrones de desigualdad socio‑ambiental. El enfoque es descriptivo y no permite inferir causalidad, pero los resultados son coherentes con la literatura sobre distribución desigual de presiones ambientales.